In [ ]:
import os
from dotenv import load_dotenv
import boto3
import json
import numpy as np
import re
from sklearn.feature_extraction import DictVectorizer
import s3fs
from sentence_transformers import SentenceTransformer

In [2]:
os.chdir("../")

In [3]:
from etl.transform import *

In [4]:
load_dotenv()

bucket_name = os.getenv("AWS_RAW_DATA")

df = extract_from_s3()

In [ ]:
new_df = df.dropna()


'\ndet = set(["hand wash", "machine", "dry clean", "cold washing", "delicate cycle", "gentle cycle", \n           "wash cold", "wash hot", "wash warm", "washable", "specialist clean", "specialist care", "wash at", "spot clean", \n           "professional clean", "delicate wash", "cold wash", "professional textile care", \n           "professional leather cleaning", "specialized care", "leather specialist", "specialist leather"])\nnew_df["new"] = [", ".join([word.lower() for word in row_list for sub in det if sub in word.lower()]) for row_list in new_df["details"]]\n\n'

In [6]:
links, data = transform_data(new_df)

# Figure out best solution for imbalanced dataset

In [ ]:
rahrah["composition"] = rahrah["composition"].str.strip()

In [ ]:
def fabric_extractor(data):
    fabric_dict = {}
    new_dict = {}

    fiber_abb = {"ac": "acetate", "ca":"acetate", "cmd": "modal", "co": "cotton", "cta": "acetate",
            "cu": "cupro", "cup": "cupro", "cv": "viscose", "ea": "elastane", "el": "elastane",
            "hl": "linen", "li": "linen", "ma": "acrylic", "mo": "modal", "me": "metallic", "ny": "polyamide",
            "pe": "polyester", "pes": "polyester", "pet": "polyester", "pm": "polyester", "pu": "polyester",
            "ra": "ramie", "se": "silk", "ta": "acetate", "vi": "viscose", "wa": "wool", "wg": "wool", "wk": "wool",
            "wl": "wool", "wm": "wool", "wp": "wool", "ws": "wool", "wy": "wool", "wv": "wool", "wo": "wool",
            "wu": "wool", "wb": "wool","pl":"polyester"
            }

    fabric_subs = {"viscose":"viscose", "rayon":"viscose", "spandex":"elastane", "elastane":"elastane", "elastan":"elastane", "tane":"elastane", "alastane":"elastane", "polytrimethylane":"polyester",
               "elasane":"elastane", "elaste":"elastane", "flax":"linen", "linen": "linen", "nylon":"polyamide", "amid":"polyamide", "polia":"polyamide", "terell":"polyester", "elasto":"polyester", 
               "cotton": "cotton", "metal": "metallic", "acetate":"acetate", "modal":"modal", "cupro":"cotton", "modacrylic":"acrylic", "acry": "acrylic","silk":"silk",
               "poly":"polyester", "lurex":"polyester", "wool":"wool", "mohair":"wool", "cashmere":"wool", "merino":"wool", "alpaca":"wool", "seta":"silk", "sisal":"linen",
               "yak":"wool", "angora":"wool", "vicuna":"wool", "llama":"wool", "camel":"wool", "guanaco":"wool", "beaver":"wool", "crepe":"polyester", "satin":"polyester", 
               "korean organza":"polyester", "organza":"silk", "ramie":"linen", "suede":"leather", "leather":"leather", "goose":"feathers", "down":"feathers", "feather":"feathers",
               "bemberg":"cotton", "lycra": "elastane","lyra":"elastane", "acette":"acetate", "ctn":"cotton", "zamac":"metallic", "agnello":"wool", "denim":"cotton",
               "shearling":"leather", "glass":"glass", "polyester":"polyester", "circulose":"cotton", "mesh":"polyester", "lyocell":"lyocell", "tencel":"lyocell", "microtencel":"lyocell",
               "skin":"leather", "pwu":"polyester","lamb":"leather", "creme":"cotton", "elit":"acrylic", "jersey":"polyester","stretch":"polyester","laine":"wool","solvron":"wool",
               "crochet":"cotton","cord":"cotton", "poplin":"cotton","pliss":"polyester","nappa":"leather", "aluminium":"metallic","hemp":"linen","spa":"spandex",
               "brass":"metallic","wax":"cotton","steel":"metal","chaguar":"linen","taffeta":"polyester","econyl":"polyester","poli":"polyester","aluminum":"metallic", "elastan":"elastane", "arcy":"acrylic"
               }

    matches = re.findall(r"(\d+(?:\.\d+)?)%\s*([\w\s-]+?)(?=\d+%|$)", data)

    total_pct = 0
    for pct_string, fabric in matches:
        pct = float(pct_string)
        fabric = fabric.strip()
        if "trochus niloticus" in fabric:
            continue
            
        fabric_dict[fabric] = fabric_dict.get(fabric, 0) + pct
        #total_pct += pct

    for fab_string, pct in fabric_dict.items():

        if fab_string in fiber_abb.keys():
            raw_fiber = fiber_abb[fab_string]
        elif any(key in fab_string for key in fabric_subs):
            for key, val in fabric_subs.items():
                if key in fab_string:
                    raw_fiber = val
                    break
                    
        else:
            raw_fiber = "other"

        new_dict[raw_fiber] = new_dict.get(raw_fiber, 0) + pct
        total_pct += pct


    if total_pct > 100:
        for fabric in new_dict:
            #scaled = (fabric_dict[fabric] / total_pct) * 100
            new_dict[fabric] = int(round((new_dict[fabric] / total_pct) * 100))

    return new_dict

#test2["composition"] = test2["composition"].str.replace(r"[\[\],']|;\s.*|Composition:\s", " ", regex=True)
t = rahrah["composition"].apply(fabric_extractor).tolist()

